# 🧰 The AI Toolbox

In this session you use **real AI models** in just a few lines of Python, then combine them into a small tool of your own, and **test whether it really works**.

You do **not** need to understand how the models work inside. You need to learn how to *use* them and *check* them.

**Repo:** `PUT-YOUR-REPO-URL-HERE`

### How we work
- 👥 **Work in pairs.** One types (driver), one thinks along (navigator). **Swap every 15 minutes.**
- ▶️ Run a cell with `Shift + Enter`. Run cells **from top to bottom**.
- 🔮 Every round has the same four steps: **Predict, Run, Change, Test**.
- 💥 Breaking things is the goal. Write down what surprises you.

### 🆘 When it breaks
| What you see | What it usually means |
|---|---|
| `NameError: name 'sentiment' is not defined` | You skipped a cell. Scroll up and run the cells above first. |
| A red error | Read **only the last line**. It tells you what's wrong. |
| `SyntaxError` on a line with `____` | That's a blank you still need to fill in. |
| Download is slow / Colab disconnected | Wait a minute. If disconnected: *Runtime > Run all*. |
| Still stuck after 5 minutes | Ask your partner, then the next pair, then the teacher. |

## Setup (run once, takes 1-2 minutes)

In [ ]:
!pip install -q "transformers<5" torch sentencepiece gradio

In [ ]:
from transformers import pipeline
print("Ready!")

## Python you need today (one screen)

```python
name = "Sam"                     # a VARIABLE: a box with a name that holds something
fruits = ["apple", "pear"]       # a LIST: several things in order
fruits[0]                        # -> "apple"   (counting starts at 0!)

person = {"name": "Sam", "age": 19}   # a DICTIONARY: labels + values
person["name"]                   # -> "Sam"     (look up a value by its label)

for fruit in fruits:             # a LOOP: do something for every item
    print(fruit)

def shout(text):                 # a FUNCTION: your own reusable mini-program
    return text.upper()
```
That's all. Everything else we explain when it shows up.

## What is a "pipeline"?

A **pipeline** is a ready-made AI tool. You load it once (slow), then use it as often as you like (fast):

```python
tool = pipeline("task", model="model-name")   # load once
result = tool("some text")                     # use many times
```

> ⚠️ Two things to remember: the model doesn't *understand* text, it recognises **patterns** it saw in training. And its confidence score ("0.90") does **not** mean "90% chance it's right".

---
# Round 1: Sentiment 😀😡
*Is a text positive or negative?*

### 🔮 Predict
What do you think the next cell prints for each sentence? Write your guess down first.

In [ ]:
# Load the tool (only do this once)
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Use the tool on one sentence
result = sentiment("I absolutely love this phone!")
print(result)

### 🔍 Look at the result
It looks like this: `[{'label': 'POSITIVE', 'score': 0.99}]`

- The `[ ]` means a **list** (with one item inside).
- The `{ }` means a **dictionary** (labels + values).
- So `result[0]` picks the first item, and `["label"]` picks the label from it.

In [ ]:
first_item = result[0]            # pick the dictionary out of the list
print(first_item["label"])        # the answer: POSITIVE or NEGATIVE
print(first_item["score"])        # how confident the model is (0 to 1)

### ✏️ Change
Try 5 sentences of your own. Can you find one the model gets **wrong**? (Try sarcasm, negations like "not bad", or Dutch.)

### ✅ Test: write your expectation BEFORE you run
Rule: first decide what you expect, **then** run. Otherwise you're just agreeing with whatever the model says.

In [ ]:
# Each item is (sentence, what YOU expect). Add your own rows!
test_set = [
    ("I love it!", "POSITIVE"),
    ("This is terrible.", "NEGATIVE"),
    ("Not bad at all!", "POSITIVE"),
    ("Great, another delay. Just what I needed.", "NEGATIVE"),
]

for sentence, expected in test_set:               # repeat for every row
    answer = sentiment(sentence)[0]["label"]      # ask the model
    if answer == expected:
        print("✅", sentence)
    else:
        print("❌", sentence, "| expected", expected, "but got", answer)

---
# Round 2: Translation 🇳🇱 ➜ 🇬🇧
*Dutch in, English out.* Most AI tools work best on **English**, so translating first is a useful trick.

### 🔮 Predict
What English do you expect for the sentence below?

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-nl-en")

result = translator("De bibliotheek sluit vanavond om negen uur.")
print(result)
print(result[0]["translation_text"])    # just the text, same idea as before: [0] then a label

### ✏️ Change + ✅ Test
Translate 3 Dutch sentences of your own (try slang, or a long one). For translation there is no single right answer, so you **compare by eye**: write your own English version first, then compare.

In [ ]:
dutch_sentences = [
    "Ik heb honger.",
    "Het is hier gezellig!",
    # add your own
]

for sentence in dutch_sentences:
    english = translator(sentence)[0]["translation_text"]
    print(sentence, "->", english)

---
# Round 3: Finding names (NER) 🕵️
*Finds people (`PER`), organisations (`ORG`) and places (`LOC`) in a text.*

### 🔮 Predict
Which names and places will it find in this sentence?

In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

text = "Satya Nadella of Microsoft visited Amsterdam with Mark Rutte."
found = ner(text)                       # a list: one dictionary per name found

for item in found:                      # look at each name found
    print(item["entity_group"], "->", item["word"])

### ✏️ Change + ✅ Test
Use a news sentence of your own. Does it find everything? Does it ever label something wrongly (e.g. a company as a person)? Does it work on a Dutch sentence?

---
# Round 4: Summaries in 3 lines 📝
*Long text in, short text out.*

The model decides the length in "tokens" (word pieces), not in lines. So we wrote a small helper that keeps the first 3 sentences.

In [ ]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

# HELPER: you only need to USE this, not understand it.
# It summarizes the text, then keeps at most 3 sentences.
def summarize_3_lines(text):
    summary = summarizer(text, max_length=80, min_length=25, do_sample=False)[0]["summary_text"]
    sentences = summary.split(". ")             # cut into sentences
    return ". ".join(sentences[:3])             # keep the first 3

### 🔮 Predict: how short will the summary be?

In [ ]:
article = '''
The city council of Utrecht announced on Monday that it will add 5,000 new bicycle parking spots
around the central station by 2028. The project, which costs around 30 million euros, is meant to
reduce the number of abandoned bikes and make the station area safer for pedestrians. Local
businesses welcomed the plan, but some residents worry about construction noise during the next
two years. The council said it will hold public meetings every month to collect feedback.
'''

print(summarize_3_lines(article))

### ✏️ Change + ✅ Test
Paste in an English article of your own (a news site works well). Then check:

In [ ]:
my_summary = summarize_3_lines(article)

# A check prints ✅ or ❌ depending on whether the statement is True
def check(name, statement):
    if statement:
        print("✅", name)
    else:
        print("❌", name)

check("summary is shorter than the original", len(my_summary) < len(article))
check("summary is not empty", len(my_summary) > 0)

## 🛑 Checkpoint
Show your partner (or the teacher) **one thing the model got wrong**. Explain in one sentence *why* you think it went wrong. Then continue.

---
# Warm-up: chain two tools ⛓️
The output of one tool can be the input of the next. Fill in the blanks (`____`). The hints are in the comments.

In [ ]:
dutch_review = "Het eten was heerlijk en de bediening was erg vriendelijk!"

# Step 1: translate to English
english_review = translator(dutch_review)[0]["____"]     # hint: which label did we use before?

# Step 2: find the sentiment of the English text
answer = sentiment(____)[0]["label"]                      # hint: what goes in the brackets?

print(answer)

---
# 🏗️ Your challenge: pick ONE

Each challenge is a **function**: text goes in, a result comes out. Then you **test it** with at least 3 checks of your own.

| | Challenge | Level |
|---|---|---|
| **A** | **Review Analyzer** 🛍️: give a shop manager the sentiment, a 3-line summary and the names in one review | ⭐ start here |
| **B** | **Dutch News Digest** 📰: Dutch article in, 3-line **English** summary + names out | ⭐⭐ (you must chain tools) |
| **C** | **Model Tester** 🔬: build a test set of 20 sentences and measure how often the sentiment model is right | ⭐⭐ (for testers) |

### Challenge A: Review Analyzer
Fill in the `TODO`s.

In [ ]:
def analyze_review(review):
    # Step 1: don't crash on empty input
    if review == "":
        return {"error": "Please give me a review."}

    # Step 2: use the tools (you made all of these before!)
    mood = None        # TODO: sentiment of the review (just the label)
    summary = None     # TODO: summarize_3_lines(...)
    names = None       # TODO: use ner(...) and collect the names

    # Step 3: return everything in one dictionary
    return {"mood": mood, "summary": summary, "names": names}


review = "I ordered a Samsung tablet from Bol.com in Utrecht. It arrived broken and support was slow. Very disappointed."
print(analyze_review(review))

In [ ]:
# ✅ Tests: make these pass, then add 3 of your own
result = analyze_review(review)

check("mood is NEGATIVE", result["mood"] == "NEGATIVE")
check("there is a summary", result["summary"] is not None)
check("empty input gives an error, not a crash", "error" in analyze_review(""))

# ➕ YOUR OWN CHECKS (at least 3). Think: what would a user type that you did NOT expect?

### Challenge B: Dutch News Digest
The summary and name tools only understand English. So what has to happen **first**?
Start with a short Dutch text:

In [ ]:
dutch_article = '''
De gemeente Utrecht gaat voor 2028 vijfduizend extra fietsparkeerplekken aanleggen bij het centraal station.
Het project kost ongeveer dertig miljoen euro. Lokale ondernemers zijn blij met het plan,
maar sommige bewoners maken zich zorgen over de bouwoverlast.
'''

# TODO: 1) translate  2) summarize  3) find the names
# Tip: the translator can only handle short texts. Translate sentence by sentence if it fails:
# sentences = dutch_article.split(". ")

**Think about it:** what happens if you summarize *before* you translate? Try both and compare.
Then write 3 checks (for example: is the summary really English? Is it max 3 sentences?).

### Challenge C: Model Tester
You don't build a tool, you **test** one. Extend the test set to **20 sentences** (include sarcasm, negation, mixed feelings). Then count how many the model gets right.

In [ ]:
test_set = [
    ("I love this!", "POSITIVE"),
    ("This is awful.", "NEGATIVE"),
    # TODO: add 18 more
]

correct = 0
for sentence, expected in test_set:
    if sentiment(sentence)[0]["label"] == expected:
        correct = correct + 1               # count this one

print("Right:", correct, "out of", len(test_set))
print("Accuracy:", correct / len(test_set) * 100, "%")

Now pick your **3 most interesting failures**. Why do you think the model got them wrong?

---
# 🚀 Stretch goal: make it a web app with Gradio

Gradio turns a function into a web page. Here is the smallest example:

In [ ]:
import gradio as gr

def mood_checker(text):
    return sentiment(text)[0]["label"]

app = gr.Interface(fn=mood_checker, inputs="text", outputs="text", title="Mood Checker")
app.launch()

### Your mission
1. Wrap **your own function** (from the challenge) in a Gradio app.
2. Then invent **your own concept**. Ideas:

| Idea | Uses |
|---|---|
| 🎓 Study-notes shrinker | summary + names |
| 🌍 Dutch news for expats | translate + summary |
| 🎭 Tone checker before you send an angry email | sentiment |
| 🕵️ "Who is mentioned?" | names |
| 🔬 Model arena: two sentiment models side by side | different `model=` names |

More models: **https://huggingface.co/models** (pick a task, copy the model name into `pipeline(...)`).

### Rules
- It runs.
- You tested it with **5 inputs**, of which **2 try to break it**.
- You can say in **one sentence** who would use it and why.

---
# 📸 Hand in
1. A screenshot of your tool working.
2. Your **best failure** (a screenshot and one sentence on why it failed).
3. One thing you'd change with one more hour.